# Amazon ML Challenge 2026 — Track 01: Comprehensive Data Audit
**Objective:** Audit, profile, and validate all 7 training and test datasets.
- 1. Dataset row counts & file sizes
- 2. Column schema & data type validation
- 3. Missing-value summary (absolute counts & percentages)
- 4. Duplicate statistics & entity ID uniqueness
- 5. Country distributions & multi-lingual script inspection
- 6. Ground truth analysis (cardinality of matches per query)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Setup project root and environment
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import src

print("Environment ready! Using src package from:", PROJECT_ROOT)


## 1. Dataset Row Counts & Overview
We inspect all 7 TSV files across `data/train/` and `data/test/`.
Because datasets contain millions of rows (~26.4M rows total), we report row counts and column schemas.


In [ ]:
# Gather high-level summaries across train and test sets
train_summary = src.get_dataset_summary(split="train", nrows=100000)
print("=== TRAIN DATASETS SUMMARY (Sample of 100k rows each for audit) ===")
display(train_summary)


In [ ]:
# Exact line/row counts across all files
import subprocess

result = subprocess.run(["wc", "-l", "data/train/train_source1.tsv", "data/train/train_source2.tsv",
                         "data/train/train_source3.tsv", "data/train/train_ground_truth.tsv",
                         "data/test/test_source1.tsv", "data/test/test_source2.tsv", "data/test/test_source3.tsv"],
                        capture_output=True, text=True)
print(result.stdout)


## 2. Column Information & Schema
Each source file contains:
- `entity_id`: Unique identifier (e.g. `S1-xxxx`, `S2-xxxx`, `S3-xxxx`)
- `business_name`: Business / trade name (contains multi-lingual text: English, Hindi/Devanagari, French)
- `business_address`: Full or partial physical address
- `country`: Country identifier (`US`, `India`, `France`, etc.)

Ground Truth contains:
- `source1_entity_id`: Query entity ID from Source 1
- `matched_entity_ids`: Comma-separated list of matching entity IDs from Source 2 and Source 3


In [ ]:
# View sample records from all three sources
for s in [1, 2, 3]:
    sample_df = src.load_source(split="train", source_num=s, nrows=5)
    print(f"\n--- Train Source {s} Sample Records ---")
    display(sample_df)


## 3. Missing-Value Summary
Let's analyze missingness per column across all sources.


In [ ]:
missing_report = []
for split in ["train", "test"]:
    for s in [1, 2, 3]:
        df = src.load_source(split=split, source_num=s, nrows=100000)
        n = len(df)
        for col in ["business_name", "business_address", "country"]:
            missing_count = df[col].isna().sum()
            missing_report.append({
                "split": split,
                "source": f"source_{s}",
                "column": col,
                "missing_count": missing_count,
                "missing_pct": round(missing_count / n * 100, 3)
            })

missing_df = pd.DataFrame(missing_report)
display(missing_df.pivot(index=["split", "source"], columns="column", values="missing_pct"))


## 4. Duplicate Statistics
Check for duplicate `entity_id` values within each source.


In [ ]:
for split in ["train", "test"]:
    for s in [1, 2, 3]:
        df = src.load_source(split=split, source_num=s, nrows=100000, usecols=["entity_id"])
        total = len(df)
        unique = df["entity_id"].nunique()
        print(f"[{split.upper()} Source {s}] Total: {total:,} | Unique IDs: {unique:,} | Duplicates: {total - unique}")


## 5. Country Distribution & Multi-lingual Text
Inspecting geographical distribution across sources.


In [ ]:
plt.figure(figsize=(12, 4))
for idx, s in enumerate([1, 2, 3], 1):
    df = src.load_source(split="train", source_num=s, nrows=100000, usecols=["country"])
    plt.subplot(1, 3, idx)
    df["country"].value_counts().plot(kind="bar", color="#2563eb")
    plt.title(f"Source {s} Country Distribution")
    plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 6. Ground Truth Cardinality Analysis
Analyzing the distribution of match counts per Source 1 query entity in `train_ground_truth.tsv`.


In [ ]:
gt_df = src.load_ground_truth(nrows=100000)
gt_pairs = src.parse_ground_truth_pairs(gt_df)

print(f"Loaded {len(gt_df):,} ground truth query rows, yielding {len(gt_pairs):,} total pairwise matches.")
match_counts = gt_pairs.groupby("source1_entity_id")["matched_entity_id"].count()

print("\nMatch count per query distribution:")
print(match_counts.describe())

print("\nMatch source distribution (S2 vs S3):")
print(gt_pairs["match_source"].value_counts(normalize=True))


In [ ]:
# Export Data Audit Summary artifact
output_path = PROJECT_ROOT / "outputs" / "data_audit_summary.csv"
train_summary.to_csv(output_path, index=False)
print(f"Audit summary successfully saved to: {output_path}")
